# Signifikansi Statistik — v1 vs v2 (Turnover)

Notebook ini menguji pertanyaan dari `MODEL_RESULTS.md`:

> Apakah model v2 (Soft Voting) **benar-benar** lebih baik daripada v1
> (Logistic Regression), atau selisih ROC-AUC 0,006 itu hanya kebetulan?

Dua uji yang dipakai:

1. **DeLong test** — uji perbandingan dua ROC-AUC pada data yang sama (berpasangan).
2. **Bootstrap berpasangan** — estimasi selang kepercayaan selisih ROC-AUC.

> Semua pipeline dibangun ulang **persis** seperti notebook v1/v2 (kontrak fitur,
> `REF_DATE`, hyperparameter dari `models/*/model_metadata.json`).

## 0. Import & Konfigurasi

In [1]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats as st
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve, average_precision_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)

CANDIDATE_PATHS = ["data/hr_turnover.csv", "../data/hr_turnover.csv", "hr_turnover.csv"]
DATA_PATH = next((Path(p) for p in CANDIDATE_PATHS if Path(p).exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("hr_turnover.csv tidak ditemukan.")

CWD = Path.cwd()
PROJECT_ROOT = CWD if (CWD / "data").exists() else CWD.parent

TARGET = "target_turnover"
REF_DATE = pd.Timestamp("2025-01-01")
TEST_SIZE = 0.20

sns.set_theme(style="whitegrid", context="notebook")
print("Data:", DATA_PATH)
print("Root:", PROJECT_ROOT)

Data: data/hr_turnover.csv
Root: /workspace/project/hr-intelligence-system


## 1. Kontrak Fitur (identik dengan v1, v2, dan API)

In [2]:
NUM_COLS = [
    "usia", "masa_kerja_bulan", "jam_lembur_bulanan", "log_gaji",
    "usia_saat_masuk", "ada_promosi", "bulan_sejak_promosi", "bulan_hingga_promosi",
]
CAT_COLS = ["departemen", "status_kerja", "pendidikan"]
FEATURES = NUM_COLS + CAT_COLS


def buat_fitur(d: pd.DataFrame, ref_date: pd.Timestamp = REF_DATE) -> pd.DataFrame:
    d = d.copy()
    masuk = pd.to_datetime(d["tanggal_masuk"])
    promosi = pd.to_datetime(d["tanggal_promosi"])
    d["ada_promosi"] = promosi.notna().astype(float)
    d["bulan_sejak_promosi"] = (ref_date - promosi).dt.days / 30.44
    d["bulan_hingga_promosi"] = (promosi - masuk).dt.days / 30.44
    d["usia_saat_masuk"] = d["usia"] - d["masa_kerja_bulan"] / 12
    d["log_gaji"] = np.log1p(d["gaji_bulanan"])
    fitur = d[FEATURES].copy()
    fitur[NUM_COLS] = fitur[NUM_COLS].astype(float)
    return fitur


def make_preprocessor() -> ColumnTransformer:
    numeric = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    return ColumnTransformer([
        ("num", numeric, NUM_COLS),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_COLS),
    ])


df_raw = pd.read_csv(DATA_PATH, parse_dates=["tanggal_masuk", "tanggal_promosi"])
X = buat_fitur(df_raw)
y = df_raw[TARGET].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=SEED)

print("Train:", X_train.shape, "| Test:", X_test.shape)
print("Rasio positif test:", round(float(y_test.mean()), 4))

Train: (800, 11) | Test: (200, 11)
Rasio positif test: 0.355


## 2. Bangun Model v1 & v2

Hyperparameter diambil dari metadata tiap versi agar reproduksinya konsisten:

- **v1** = Logistic Regression (`C` dari `models/v1/model_metadata.json`).
- **v2** = Soft Voting (Logistic Regression + Random Forest tuned).

In [3]:
meta_v1 = json.loads((PROJECT_ROOT / "models/v1/model_metadata.json").read_text())
meta_v2 = json.loads((PROJECT_ROOT / "models/v2/model_metadata.json").read_text())

hp_v1 = meta_v1["hyperparameter"]
hp_rf = meta_v2["hyperparameter"]["Random Forest"]
print("v1  :", hp_v1)
print("v2  :", hp_rf)

lr_v1 = LogisticRegression(C=hp_v1["C"], max_iter=2000, random_state=SEED)
rf = RandomForestClassifier(
    n_estimators=int(hp_rf["n_estimators"]),
    max_depth=hp_rf["max_depth"],
    max_features=hp_rf["max_features"],
    min_samples_leaf=int(hp_rf["min_samples_leaf"]),
    random_state=SEED, n_jobs=-1)

pipe_v1 = Pipeline([("prep", make_preprocessor()), ("clf", lr_v1)])
pipe_v2 = Pipeline([("prep", make_preprocessor()),
                    ("clf", VotingClassifier(
                        estimators=[("lr", LogisticRegression(C=hp_v1["C"], max_iter=2000, random_state=SEED)),
                                    ("rf", rf)],
                        voting="soft"))])

pipe_v1.fit(X_train, y_train)
pipe_v2.fit(X_train, y_train)

p_v1 = pipe_v1.predict_proba(X_test)[:, 1]
p_v2 = pipe_v2.predict_proba(X_test)[:, 1]
y_te = y_test.to_numpy()

print(f"ROC-AUC v1: {roc_auc_score(y_te, p_v1):.4f}")
print(f"ROC-AUC v2: {roc_auc_score(y_te, p_v2):.4f}")

v1  : {'C': 0.006026889128682512}
v2  : {'max_depth': 4, 'max_features': 'log2', 'max_leaf_nodes': None, 'min_samples_leaf': 9, 'n_estimators': 645}


ROC-AUC v1: 0.6929
ROC-AUC v2: 0.6993


## 3. DeLong Test

DeLong (1988) menguji apakah dua ROC-AUC berbeda secara signifikan, dengan
memperhitungkan korelasi karena dinilai pada data yang sama.

H0: AUC(v1) = AUC(v2). Jika p < 0,05 → selisih signifikan.

In [4]:
def _compute_midrank(x):
    J = np.argsort(x)
    Z = x[J]
    N = len(x)
    T = np.zeros(N, dtype=float)
    i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]:
            j += 1
        T[i:j] = 0.5 * (i + j - 1)
        i = j
    T2 = np.empty(N, dtype=float)
    T2[J] = T + 1
    return T2


def _fast_delong(preds, n_pos):
    """preds: (k, n) skor; n_pos: jumlah sampel positif. Mengembalikan AUC & kovarians."""
    m = n_pos
    n = preds.shape[1] - m
    pos, neg = preds[:, :m], preds[:, m:]
    k = preds.shape[0]
    tx = np.empty((k, m)); ty = np.empty((k, n)); tz = np.empty((k, m + n))
    for r in range(k):
        tx[r] = _compute_midrank(pos[r])
        ty[r] = _compute_midrank(neg[r])
        tz[r] = _compute_midrank(preds[r])
    aucs = tz[:, :m].sum(axis=1) / m / n - (m + 1.0) / 2.0 / n
    v01 = (tz[:, :m] - tx) / n
    v10 = 1.0 - (tz[:, m:] - ty) / m
    cov = np.cov(v01) / m + np.cov(v10) / n
    return aucs, np.atleast_2d(cov)


def delong_test(y_true, p_a, p_b):
    order = np.argsort(-y_true)
    n_pos = int(np.sum(y_true))
    preds = np.vstack([p_a, p_b])[:, order]
    aucs, cov = _fast_delong(preds, n_pos)
    var = cov[0, 0] + cov[1, 1] - 2 * cov[0, 1]
    z = (aucs[0] - aucs[1]) / np.sqrt(var) if var > 0 else 0.0
    p = 2 * (1 - st.norm.cdf(abs(z)))
    return float(aucs[0]), float(aucs[1]), float(z), float(p)


auc_a, auc_b, z, pval = delong_test(y_te, p_v1, p_v2)
print(f"AUC v1        : {auc_a:.4f}")
print(f"AUC v2        : {auc_b:.4f}")
print(f"Selisih (v2-v1): {auc_b - auc_a:+.4f}")
print(f"z             : {z:.4f}")
print(f"p-value       : {pval:.4f}")
print("\nKesimpulan:", "SIGNIFIKAN" if pval < 0.05 else "TIDAK signifikan (p >= 0,05)")

AUC v1        : 0.6929
AUC v2        : 0.6993
Selisih (v2-v1): +0.0064
z             : -0.4638
p-value       : 0.6428

Kesimpulan: TIDAK signifikan (p >= 0,05)


## 4. Bootstrap Berpasangan

Resampling test set 5.000 kali; tiap iterasi menghitung ROC-AUC v1 dan v2 pada
sampel yang sama, lalu selisihnya. Ini memberi selang kepercayaan selisih.

In [5]:
N_BOOT = 5000
rng = np.random.default_rng(SEED)
n = len(y_te)
deltas = np.empty(N_BOOT)
boot_a = np.empty(N_BOOT); boot_b = np.empty(N_BOOT)

for i in range(N_BOOT):
    idx = rng.integers(0, n, n)
    yb = y_te[idx]
    if yb.min() == yb.max():
        deltas[i] = np.nan; boot_a[i] = np.nan; boot_b[i] = np.nan
        continue
    a = roc_auc_score(yb, p_v1[idx]); b = roc_auc_score(yb, p_v2[idx])
    boot_a[i] = a; boot_b[i] = b; deltas[i] = b - a

deltas = deltas[~np.isnan(deltas)]
lo, hi = np.percentile(deltas, [2.5, 97.5])
p_boot = 2 * min((deltas <= 0).mean(), (deltas >= 0).mean())

print(f"Selisih ROC-AUC (v2 - v1): {deltas.mean():+.4f}")
print(f"95% CI bootstrap          : [{lo:+.4f}, {hi:+.4f}]")
print(f"p-value bootstrap (2 sisi): {p_boot:.4f}")
print("CI memuat 0 :", "YA → tidak signifikan" if lo <= 0 <= hi else "TIDAK → signifikan")

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(deltas, bins=50, color="#4C78A8", alpha=0.85)
ax.axvline(0, color="black", ls="--", lw=1.5, label="tidak ada perbedaan")
ax.axvline(deltas.mean(), color="#E45756", lw=2, label=f"rata-rata = {deltas.mean():+.4f}")
ax.set_xlabel("Δ ROC-AUC (v2 − v1)"); ax.set_ylabel("frekuensi")
ax.set_title("Distribusi selisih ROC-AUC (bootstrap berpasangan)")
ax.legend(); plt.tight_layout(); plt.show()

Selisih ROC-AUC (v2 - v1): +0.0064
95% CI bootstrap          : [-0.0214, +0.0342]
p-value bootstrap (2 sisi): 0.6596
CI memuat 0 : YA → tidak signifikan


## 5. Ringkasan & Kurva ROC

In [6]:
tabel = pd.DataFrame({
    "model": ["v1 (Logistic Regression)", "v2 (Soft Voting)"],
    "roc_auc": [auc_a, auc_b],
    "pr_auc": [average_precision_score(y_te, p_v1), average_precision_score(y_te, p_v2)],
})
tabel["selisih_vs_v1"] = tabel["roc_auc"] - auc_a

print(tabel.round(4).to_string(index=False))
print()
print(pd.Series({
    "delong_z": z, "delong_p": pval,
    "bootstrap_mean_delta": deltas.mean(),
    "bootstrap_ci_low": lo, "bootstrap_ci_high": hi,
    "bootstrap_p": p_boot,
}).round(4).to_string())

fpr1, tpr1, _ = roc_curve(y_te, p_v1)
fpr2, tpr2, _ = roc_curve(y_te, p_v2)
plt.figure(figsize=(6, 6))
plt.plot(fpr1, tpr1, label=f"v1 (AUC={auc_a:.3f})", color="#4C78A8")
plt.plot(fpr2, tpr2, label=f"v2 (AUC={auc_b:.3f})", color="#E45756")
plt.plot([0, 1], [0, 1], ls="--", color="grey", lw=1)
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title("Kurva ROC — v1 vs v2 (test set)"); plt.legend()
plt.tight_layout(); plt.show()

                   model  roc_auc  pr_auc  selisih_vs_v1
v1 (Logistic Regression)   0.6929  0.6235         0.0000
        v2 (Soft Voting)   0.6993  0.6223         0.0064

delong_z               -0.4638
delong_p                0.6428
bootstrap_mean_delta    0.0064
bootstrap_ci_low       -0.0214
bootstrap_ci_high       0.0342
bootstrap_p             0.6596


## 6. Kesimpulan

Tulis hasil otomatis berdasarkan uji di atas.

In [7]:
signifikan = (pval < 0.05) and not (lo <= 0 <= hi)
kesimpulan = (
    "v2 TERBUKTI lebih baik secara statistik" if signifikan
    else "v2 TIDAK terbukti lebih baik; selisih kemungkinan hanya noise"
)
hasil = {
    "auc_v1": round(auc_a, 4),
    "auc_v2": round(auc_b, 4),
    "delta_auc": round(float(auc_b - auc_a), 4),
    "delong_z": round(z, 4),
    "delong_p": round(pval, 4),
    "bootstrap_ci": [round(float(lo), 4), round(float(hi), 4)],
    "bootstrap_p": round(float(p_boot), 4),
    "kesimpulan": kesimpulan,
    "rekomendasi": (
        "Tetap pakai model yang lebih sederhana & lebih adil (v1) sampai ada "
        "bukti tambahan pada data yang lebih besar."
        if not signifikan else
        "v2 layak dipakai; lanjutkan dengan mitigasi bias (lihat notebook 08)."
    ),
}
out = PROJECT_ROOT / "models" / "significance_report.json"
out.write_text(json.dumps(hasil, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps(hasil, indent=2, ensure_ascii=False))
print("\nDitulis:", out)

{
  "auc_v1": 0.6929,
  "auc_v2": 0.6993,
  "delta_auc": 0.0064,
  "delong_z": -0.4638,
  "delong_p": 0.6428,
  "bootstrap_ci": [
    -0.0214,
    0.0342
  ],
  "bootstrap_p": 0.6596,
  "kesimpulan": "v2 TIDAK terbukti lebih baik; selisih kemungkinan hanya noise",
  "rekomendasi": "Tetap pakai model yang lebih sederhana & lebih adil (v1) sampai ada bukti tambahan pada data yang lebih besar."
}

Ditulis: /workspace/project/hr-intelligence-system/models/significance_report.json
